# PII redaction at inference time with audit trail

A summarization service processes customer-support emails, some of which contain PII (emails, phone numbers, SSNs, credit-card numbers). Compliance wants PII redacted in outputs AND the redaction auditable. You have 75 minutes to build a two-stage pipeline: pre-processing with AWS Comprehend PII detection on a 100-email synthetic dataset, output-stage with Bedrock Guardrails to catch stealth PII the regex misses, and an audit log to an S3 bucket with object lock for tamper evidence.

**Duration:** about 75 minutes of work in a 105-minute session window.

**Cost preamble.** Cheapest lab in this sub-track. Comprehend at $0.0001 per 100 characters lands near 20 cents on this corpus; Bedrock Guardrails on input+output is another 15 cents; Haiku summarization is a dime. About 60 cents on a clean run, $1.20 if you re-process the dataset while debugging. S3 + object lock is free.

In [ ]:
# NBVAL_SKIP
# Pinned dependencies. boto3 covers Comprehend, Bedrock, and S3 in one SDK.

!pip install --quiet labrun-checks==0.7.0 anthropic==0.42.0 boto3==1.35.0

In [ ]:
# Imports and per-lab constants.

import atexit
import csv
import getpass
import hashlib
import json
import os
import re
import sys
import time
import uuid
from datetime import datetime, timezone

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupAdapter,
    CleanupResource,
)

LAB_ID = "pii-redaction-with-audit-trail"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "us-east-1"

INPUT_BUCKET_BASE = f"labrun-{LAB_ID}-input"
AUDIT_BUCKET_BASE = f"labrun-{LAB_ID}-audit"
GUARDRAIL_NAME = f"labrun-{LAB_ID}-guardrail"
TRACES_PATH = "/content/redaction_traces.csv"

NUM_EMAILS = 100  # 50 with embedded PII, 50 clean, 5 of the PII set are stealth.

PII_ENTITY_TYPES = [
    "EMAIL",
    "PHONE",
    "SSN",
    "CREDIT_DEBIT_NUMBER",
    "NAME",
    "ADDRESS",
    "USERNAME",
    "BANK_ACCOUNT_NUMBER",
]

ANTHROPIC_HAIKU = "claude-haiku-4-5-20250930"

In [ ]:
# NBVAL_SKIP
# BYOK: session token, AWS keys, Anthropic key. Smoke-test STS, Comprehend
# DetectDominantLanguage on a trivial string, Bedrock ListFoundationModels,
# Anthropic Haiku ping. Register session.

import boto3
import anthropic
from botocore.exceptions import ClientError

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
AWS_ACCESS_KEY_ID = getpass.getpass("AWS_ACCESS_KEY_ID: ").strip()
AWS_SECRET_ACCESS_KEY = getpass.getpass("AWS_SECRET_ACCESS_KEY: ").strip()
ANTHROPIC_API_KEY = getpass.getpass("ANTHROPIC_API_KEY: ").strip()

if not all([AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, ANTHROPIC_API_KEY]):
    print("AWS keys + Anthropic key are all required. Re-run this cell.")
    raise SystemExit(1)

os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
os.environ["AWS_DEFAULT_REGION"] = REGION
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

sts = boto3.client("sts", region_name=REGION)
try:
    ACCOUNT_ID = sts.get_caller_identity()["Account"]
    print(f"AWS credentials validated. Account: {ACCOUNT_ID}.")
except ClientError as exc:
    print(f"AWS credentials rejected: {exc!r}")
    raise SystemExit(1)

INPUT_BUCKET = f"{INPUT_BUCKET_BASE}-{ACCOUNT_ID}"
AUDIT_BUCKET = f"{AUDIT_BUCKET_BASE}-{ACCOUNT_ID}"

s3 = boto3.client("s3", region_name=REGION)
comprehend = boto3.client("comprehend", region_name=REGION)
bedrock = boto3.client("bedrock", region_name=REGION)
bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)

try:
    comprehend.detect_dominant_language(Text="hello world")
    print("Comprehend reachable.")
except ClientError as exc:
    print(f"Comprehend rejected: {exc!r}")
    raise SystemExit(1)

try:
    bedrock.list_foundation_models()
    print("Bedrock reachable.")
except ClientError as exc:
    print("Bedrock rejected. Enable Bedrock model access for the account from the AWS console")
    print("(Bedrock > Model access) before retrying. The Guardrails service itself does not")
    print("require model access, but the client init wants the regional endpoint to resolve.")
    print(f"  {exc!r}")
    raise SystemExit(1)

anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
try:
    _ping = anthropic_client.messages.create(
        model=ANTHROPIC_HAIKU,
        max_tokens=8,
        messages=[{"role": "user", "content": "reply with: ok"}],
    )
    print(f"Anthropic credentials validated. Haiku replied: {_ping.content[0].text!r}")
except Exception as exc:
    print(f"Anthropic key rejected: {exc!r}")
    raise SystemExit(1)

register(
    session_token=session_token,
    lab_id=LAB_ID,
    credentials={
        "region": REGION,
        "account_id": ACCOUNT_ID,
        "input_bucket": INPUT_BUCKET,
        "audit_bucket": AUDIT_BUCKET,
    },
)
print(f"Session registered. Input bucket: {INPUT_BUCKET}. Audit bucket: {AUDIT_BUCKET}.")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, adapter, atexit safety net, orphan scan.
# IMPORTANT: object lock objects in the audit bucket require a governance
# bypass to delete. The adapter handles that automatically.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="local_file",
        id=TRACES_PATH,
        region="local",
        cli_delete_command=f"rm -f {TRACES_PATH}",
    ),
    CleanupResource(
        type="bedrock_guardrail",
        id=GUARDRAIL_NAME,
        region=REGION,
        cli_delete_command=f"aws bedrock delete-guardrail --guardrail-identifier {GUARDRAIL_NAME} --region {REGION}",
    ),
    CleanupResource(
        type="s3_bucket_object_locked",
        id=AUDIT_BUCKET,
        region=REGION,
        cli_delete_command=f"# manual: empty with --force-bypass-governance then delete",
    ),
    CleanupResource(
        type="s3_bucket",
        id=INPUT_BUCKET,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{INPUT_BUCKET} --force",
    ),
]


# Track the live guardrail id; create_guardrail returns an id distinct from
# its name and the adapter needs the id to delete.
LIVE = {"guardrail_id": None, "guardrail_version": None}


class PiiLabCleanupAdapter:
    """Inline adapter for S3 buckets (plain and object-locked), Bedrock
    Guardrails, and local files.

    TODO: labrun-checks 0.7.0 has no aws.bedrock_guardrail or aws.s3_object_lock
    surface; once 0.8 ships, migrate.
    """

    def delete_resource(self, credentials, resource):
        if resource.type == "local_file":
            try:
                os.remove(resource.id)
            except FileNotFoundError:
                pass
            return
        if resource.type == "s3_bucket":
            try:
                paginator = s3.get_paginator("list_objects_v2")
                for page in paginator.paginate(Bucket=resource.id):
                    if "Contents" in page:
                        s3.delete_objects(
                            Bucket=resource.id,
                            Delete={"Objects": [{"Key": o["Key"]} for o in page["Contents"]]},
                        )
                s3.delete_bucket(Bucket=resource.id)
            except ClientError as exc:
                if exc.response["Error"]["Code"] == "NoSuchBucket":
                    return
                raise
            return
        if resource.type == "s3_bucket_object_locked":
            self._delete_object_locked_bucket(resource.id)
            return
        if resource.type == "bedrock_guardrail":
            guardrail_id = LIVE.get("guardrail_id")
            if guardrail_id is None:
                # Try by name resolution: list and match.
                try:
                    resp = bedrock.list_guardrails()
                    for g in resp.get("guardrails", []):
                        if g["name"] == resource.id:
                            guardrail_id = g["id"]
                            break
                except ClientError:
                    return
            if guardrail_id:
                try:
                    bedrock.delete_guardrail(guardrailIdentifier=guardrail_id)
                except ClientError as exc:
                    if exc.response["Error"]["Code"] == "ResourceNotFoundException":
                        return
                    raise
            return

    def _delete_object_locked_bucket(self, bucket):
        try:
            # Versions, not just contents: object-locked buckets are versioned.
            paginator = s3.get_paginator("list_object_versions")
            for page in paginator.paginate(Bucket=bucket):
                victims = []
                for v in page.get("Versions", []) + page.get("DeleteMarkers", []):
                    victims.append({"Key": v["Key"], "VersionId": v["VersionId"]})
                if victims:
                    s3.delete_objects(
                        Bucket=bucket,
                        Delete={"Objects": victims},
                        BypassGovernanceRetention=True,
                    )
            s3.delete_bucket(Bucket=bucket)
        except ClientError as exc:
            code = exc.response["Error"]["Code"]
            if code in ("NoSuchBucket",):
                return
            raise

    def describe_resource(self, credentials, resource):
        if resource.type == "local_file":
            return os.path.exists(resource.id)
        if resource.type in ("s3_bucket", "s3_bucket_object_locked"):
            try:
                s3.head_bucket(Bucket=resource.id)
                return True
            except ClientError:
                return False
        if resource.type == "bedrock_guardrail":
            try:
                guardrail_id = LIVE.get("guardrail_id")
                if not guardrail_id:
                    return False
                bedrock.get_guardrail(guardrailIdentifier=guardrail_id)
                return True
            except ClientError:
                return False
        return False

    def tag_scan(self, credentials, lab_slug, region):
        try:
            rg = boto3.client("resourcegroupstaggingapi", region_name=region)
            resp = rg.get_resources(TagFilters=[{"Key": LAB_TAG_KEY, "Values": [lab_slug]}])
            return [r["ResourceARN"] for r in resp.get("ResourceTagMappingList", [])]
        except Exception:
            return []


CLEANUP_ADAPTER = PiiLabCleanupAdapter()


# Orphan scan blocks if a prior session left tagged resources.
try:
    rg = boto3.client("resourcegroupstaggingapi", region_name=REGION)
    resp = rg.get_resources(TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_ID]}])
    _orphans = [r["ResourceARN"] for r in resp.get("ResourceTagMappingList", [])]
except Exception:
    _orphans = []

if _orphans:
    print("Orphan scan found tagged resources from a prior session:")
    for a in _orphans:
        print(f"  {a}")
    print("Clean those up before re-running this lab.")
    raise SystemExit(1)


def _on_exit_cleanup():
    try:
        for r in list(CLEANUP_MANIFEST):
            try:
                CLEANUP_ADAPTER.delete_resource({}, r)
            except Exception:
                pass
    except Exception:
        pass


atexit.register(_on_exit_cleanup)
print("Manifest registered. Orphan scan clean.")

## Task 1: Generate the synthetic corpus, create the buckets, run Comprehend pre-stage

The synthetic dataset is generated in-kernel so it is deterministic. 100 emails: 50 contain PII (mix of email, phone, SSN, credit-card, name, address); 50 are clean. Inside the PII set, 5 are "stealth" cases that bypass Comprehend's regex/model (SSN without dashes inside narrative prose, credit-card number split across lines, etc.).

You create:
- input S3 bucket (plain, tagged)
- audit S3 bucket with object lock enabled AT bucket creation (cannot be enabled later) in governance mode

Then run Comprehend `detect_pii_entities` against each email, count detected entities per type, and write a row per email to `/content/redaction_traces.csv` with `email_id`, `ground_truth_pii_count`, `comprehend_detected_pii_count`, `stealth_case` boolean, plus the residual `remaining_text` after applying Comprehend offsets.

In [ ]:
# Task 1: corpus + buckets + Comprehend pre-stage.

def synth_corpus(n=NUM_EMAILS):
    """50 PII-bearing emails + 50 clean. 5 of the PII emails are stealth."""
    pii_templates = [
        ("Hi support, can you reach me at sarah.kim@example.com about ticket 4421?", 1),
        ("Call me at (415) 555-0123 between 9 and 5 PT. Thanks.", 1),
        ("My SSN on file is 123-45-6789. Please update the W-9.", 1),
        ("Card ending 4242 was declined; the full number is 4111 1111 1111 1111.", 1),
        ("Mailing address: 1234 Mission St, San Francisco CA 94103. Signed, John Doe.", 2),
        ("Hello team, just confirming my email is alex@globex.io and phone 510-555-9988.", 2),
    ]
    stealth_templates = [
        ("To verify identity our records show social-security one-two-three four-five "
         "six-seven-eight-nine, please confirm.", 1),
        ("Card no four one one one space one one one one space one one one one space "
         "one one one one declined again.", 1),
        ("Reach me at sarah dot kim at example dot com for the refund.", 1),
        ("Phone four one five five five five zero one two three, mornings only.", 1),
        ("SSN one two three four five six seven eight nine, all run together.", 1),
    ]
    clean_template = "Following up on ticket {n}. No urgency, just confirming receipt."

    emails = []
    for i in range(n // 2):
        if i < 5:
            text, count = stealth_templates[i % len(stealth_templates)]
            emails.append({"id": f"em_{i:03d}", "text": text, "gt_pii": count, "stealth": True})
        else:
            text, count = pii_templates[i % len(pii_templates)]
            emails.append({"id": f"em_{i:03d}", "text": text, "gt_pii": count, "stealth": False})
    for i in range(n // 2, n):
        emails.append({
            "id": f"em_{i:03d}",
            "text": clean_template.format(n=4000 + i),
            "gt_pii": 0,
            "stealth": False,
        })
    return emails


corpus = synth_corpus(NUM_EMAILS)


def create_buckets():
    # Input bucket: plain, tagged. us-east-1 takes no LocationConstraint.
    # YOUR CODE: create the input bucket with name INPUT_BUCKET, then call
    # s3.put_bucket_tagging with TagSet=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
    # YOUR CODE: create the audit bucket with name AUDIT_BUCKET, passing
    # ObjectLockEnabledForBucket=True. Then put_object_lock_configuration with
    # ObjectLockEnabled="Enabled" and Rule={"DefaultRetention": {"Mode":
    # "GOVERNANCE", "Days": 1}}. Tag the audit bucket too.
    raise NotImplementedError("Replace with the two create_bucket calls.")


create_buckets()
print(f"Input bucket: {INPUT_BUCKET}.")
print(f"Audit bucket: {AUDIT_BUCKET} (object lock GOVERNANCE 1 day).")


# Upload corpus to input bucket as a single JSONL object.
s3.put_object(
    Bucket=INPUT_BUCKET,
    Key="emails.jsonl",
    Body="\n".join(json.dumps(e) for e in corpus).encode(),
)


def comprehend_detect(text):
    # YOUR CODE: call comprehend.detect_pii_entities(Text=text, LanguageCode="en").
    # Return resp["Entities"] (a list of {Type, Score, BeginOffset, EndOffset}).
    raise NotImplementedError("Replace with the Comprehend detect call.")


with open(TRACES_PATH, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["email_id", "gt_pii", "stealth", "comprehend_detected", "remaining_text"])
    for e in corpus:
        ents = comprehend_detect(e["text"])
        # Replace each entity with the redacted placeholder for downstream stage.
        text = e["text"]
        for ent in sorted(ents, key=lambda x: -x["BeginOffset"]):
            text = text[:ent["BeginOffset"]] + f"[{ent['Type']}]" + text[ent["EndOffset"]:]
        w.writerow([e["id"], e["gt_pii"], int(e["stealth"]), len(ents), text])

print(f"Wrote pre-stage traces to {TRACES_PATH}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Comprehend caught >= 90% of PII-bearing non-stealth emails.
# The trace rows are the ground truth source.


def checkpoint_1(session):
    if not os.path.exists(TRACES_PATH):
        return CheckpointResult(status="fail", error_reason=f"{TRACES_PATH} missing.")
    with open(TRACES_PATH) as f:
        rows = list(csv.DictReader(f))
    if len(rows) < NUM_EMAILS:
        return CheckpointResult(
            status="fail",
            error_reason=f"Expected {NUM_EMAILS} trace rows, got {len(rows)}.",
        )
    pii_rows = [r for r in rows if int(r["gt_pii"]) > 0 and int(r["stealth"]) == 0]
    caught = sum(1 for r in pii_rows if int(r["comprehend_detected"]) >= 1)
    if not pii_rows:
        return CheckpointResult(status="fail", error_reason="Synth corpus produced no non-stealth PII rows.")
    recall = caught / len(pii_rows)
    if recall < 0.90:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Comprehend recall on non-stealth PII rows is {recall:.0%}; expected >= 90%. "
                f"Most likely cause: detect_pii_entities not called or LanguageCode wrong."
            ),
        )
    return CheckpointResult(status="pass")


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

Two buckets, one with object lock. Object lock is the gotcha: it cannot be enabled after bucket creation. The `create_bucket` call needs `ObjectLockEnabledForBucket=True` and a follow-up `put_object_lock_configuration` to set the retention mode.

</details>

<details><summary>Hint 2 (direction)</summary>

```
s3.create_bucket(Bucket=INPUT_BUCKET)
s3.put_bucket_tagging(Bucket=INPUT_BUCKET, Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]})

s3.create_bucket(Bucket=AUDIT_BUCKET, ObjectLockEnabledForBucket=True)
s3.put_object_lock_configuration(
    Bucket=AUDIT_BUCKET,
    ObjectLockConfiguration={
        "ObjectLockEnabled": "Enabled",
        "Rule": {"DefaultRetention": {"Mode": "GOVERNANCE", "Days": 1}},
    },
)
s3.put_bucket_tagging(Bucket=AUDIT_BUCKET, Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]})
```

For Comprehend:

```
resp = comprehend.detect_pii_entities(Text=text, LanguageCode="en")
return resp["Entities"]
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Full `create_buckets`:

```python
def create_buckets():
    try:
        s3.create_bucket(Bucket=INPUT_BUCKET)
    except ClientError as exc:
        if exc.response["Error"]["Code"] not in ("BucketAlreadyOwnedByYou",):
            raise
    s3.put_bucket_tagging(
        Bucket=INPUT_BUCKET,
        Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
    )
    try:
        s3.create_bucket(Bucket=AUDIT_BUCKET, ObjectLockEnabledForBucket=True)
    except ClientError as exc:
        if exc.response["Error"]["Code"] not in ("BucketAlreadyOwnedByYou",):
            raise
    s3.put_object_lock_configuration(
        Bucket=AUDIT_BUCKET,
        ObjectLockConfiguration={
            "ObjectLockEnabled": "Enabled",
            "Rule": {"DefaultRetention": {"Mode": "GOVERNANCE", "Days": 1}},
        },
    )
    s3.put_bucket_tagging(
        Bucket=AUDIT_BUCKET,
        Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
    )


def comprehend_detect(text):
    resp = comprehend.detect_pii_entities(Text=text, LanguageCode="en")
    return resp["Entities"]
```

</details>

**Wallet check.** Comprehend at $0.0001 per 100 characters: ~20 cents for 100 emails averaging ~200 chars each. Bucket creation is free. Cumulative spend: about 20 cents.

## Task 2: Create the Bedrock Guardrail and catch the stealth PII

Bedrock Guardrails operate on text input AND output. Configure a guardrail that flags every entity in `PII_ENTITY_TYPES` and applies `ANONYMIZE`. Run the 5 stealth cases through `apply_guardrail`; the residual_text from Task 1 still contains the stealth PII (Comprehend missed it). The Guardrail catches it at the output stage.

Append a row per stealth case to the traces CSV with `output_stage_flagged` set to True/False.

In [ ]:
# Task 2: Bedrock Guardrails.

def create_guardrail():
    # YOUR CODE: call bedrock.create_guardrail with:
    #   name=GUARDRAIL_NAME,
    #   blockedInputMessaging="Input blocked due to PII.",
    #   blockedOutputsMessaging="Output blocked due to PII.",
    #   piiEntitiesConfig=[{"type": t, "action": "ANONYMIZE"} for t in PII_ENTITY_TYPES],
    #   ... etc. Return (guardrail_id, version).
    # Note: the API requires version "DRAFT" or a specific version number for
    # apply_guardrail; create_guardrail returns the id; use "DRAFT" for the
    # subsequent apply call.
    raise NotImplementedError("Replace with bedrock.create_guardrail.")


guardrail_id, guardrail_version = create_guardrail()
LIVE["guardrail_id"] = guardrail_id
LIVE["guardrail_version"] = guardrail_version
print(f"Guardrail created: id={guardrail_id} version={guardrail_version}")


def apply_guardrail(text):
    # YOUR CODE: call bedrock_runtime.apply_guardrail with:
    #   guardrailIdentifier=guardrail_id,
    #   guardrailVersion=guardrail_version,
    #   source="OUTPUT",
    #   content=[{"text": {"text": text}}].
    # Return the response. The response's "action" field is GUARDRAIL_INTERVENED
    # when the guardrail anonymized or blocked the content; otherwise NONE.
    raise NotImplementedError("Replace with bedrock_runtime.apply_guardrail.")


stealth_results = []
with open(TRACES_PATH) as f:
    rows = list(csv.DictReader(f))

for row in rows:
    if int(row["stealth"]) != 1:
        continue
    resp = apply_guardrail(row["remaining_text"])
    flagged = resp.get("action") == "GUARDRAIL_INTERVENED"
    stealth_results.append({"email_id": row["email_id"], "output_flagged": flagged})

print(f"Stealth cases processed: {len(stealth_results)}")
print(f"Flagged by Guardrail: {sum(r['output_flagged'] for r in stealth_results)}")

# Persist stealth flags into the trace file.
with open(TRACES_PATH) as f:
    rows = list(csv.DictReader(f))
fieldnames = list(rows[0].keys()) + (["output_stage_flagged"] if "output_stage_flagged" not in rows[0] else [])
flag_by_id = {r["email_id"]: r["output_flagged"] for r in stealth_results}
for r in rows:
    if int(r["stealth"]) == 1:
        r["output_stage_flagged"] = flag_by_id.get(r["email_id"], False)
    else:
        r["output_stage_flagged"] = ""
with open(TRACES_PATH, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    for r in rows:
        w.writerow(r)

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: at least 4 of the 5 stealth cases caught at the output stage.


def checkpoint_2(session):
    if LIVE.get("guardrail_id") is None:
        return CheckpointResult(status="fail", error_reason="Guardrail was not created; LIVE.guardrail_id is None.")
    with open(TRACES_PATH) as f:
        rows = list(csv.DictReader(f))
    stealth = [r for r in rows if int(r["stealth"]) == 1]
    flagged = sum(1 for r in stealth if str(r.get("output_stage_flagged", "")).lower() == "true")
    if flagged < 4:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Only {flagged} of {len(stealth)} stealth cases flagged at output stage. "
                f"Confirm apply_guardrail was called with source='OUTPUT' and the guardrail "
                f"config anonymizes the relevant PII types."
            ),
        )
    return CheckpointResult(status="pass")


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

Two API calls. `create_guardrail` builds the config; `apply_guardrail` runs text through it. Both live on different clients: `bedrock` (control plane) for create/delete, `bedrock-runtime` for apply.

</details>

<details><summary>Hint 2 (direction)</summary>

```
resp = bedrock.create_guardrail(
    name=GUARDRAIL_NAME,
    blockedInputMessaging="Input blocked due to PII.",
    blockedOutputsMessaging="Output blocked due to PII.",
    piiEntitiesConfig=[{"type": t, "action": "ANONYMIZE"} for t in PII_ENTITY_TYPES],
)
guardrail_id = resp["guardrailId"]
# For applying, use "DRAFT" until you publish a numbered version.
guardrail_version = "DRAFT"
```

For apply:

```
resp = bedrock_runtime.apply_guardrail(
    guardrailIdentifier=guardrail_id,
    guardrailVersion=guardrail_version,
    source="OUTPUT",
    content=[{"text": {"text": text}}],
)
```

`resp["action"]` is `"GUARDRAIL_INTERVENED"` when anything was anonymized.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
def create_guardrail():
    resp = bedrock.create_guardrail(
        name=GUARDRAIL_NAME,
        blockedInputMessaging="Input blocked due to PII.",
        blockedOutputsMessaging="Output blocked due to PII.",
        piiEntitiesConfig=[{"type": t, "action": "ANONYMIZE"} for t in PII_ENTITY_TYPES],
    )
    return resp["guardrailId"], "DRAFT"


def apply_guardrail(text):
    resp = bedrock_runtime.apply_guardrail(
        guardrailIdentifier=LIVE["guardrail_id"],
        guardrailVersion=LIVE["guardrail_version"],
        source="OUTPUT",
        content=[{"text": {"text": text}}],
    )
    return resp
```

</details>

**Wallet check.** Bedrock Guardrails: ~$0.0075 per 1K text units; 5 stealth cases is well under a penny. Plus the 100-email Comprehend pre-stage. Spent so far: under 25 cents.

## Task 3: Write audit log entries to the object-locked S3 bucket

For every PII-detected entity (across both stages), write an audit-log object to the audit bucket. Each entry stores: `entity_type`, `position` (begin offset), `sha256_hash` of the redacted value (NOT the value), `timestamp`. The validator confirms no audit log body contains PII patterns (email, phone, SSN, credit-card regexes) and that the log was written with the default object-lock retention.

In [ ]:
# Task 3: audit log entries.

ALLOWED_FIELDS = {"entity_type", "position", "sha256_hash", "timestamp", "email_id"}

# Regex sentinels the validator uses to confirm no PII leaked into the log.
PII_PATTERNS = [
    re.compile(r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b", re.IGNORECASE),  # email
    re.compile(r"\b\d{3}[-\s]?\d{2}[-\s]?\d{4}\b"),  # SSN
    re.compile(r"\b\d{3}[-.()\s]?\d{3}[-.\s]?\d{4}\b"),  # phone
    re.compile(r"\b(?:\d[ -]*?){13,19}\b"),  # cc
]


def write_audit_entry(email_id, entity_type, position, raw_value):
    # YOUR CODE: hash raw_value with hashlib.sha256(raw_value.encode()).hexdigest().
    # Build a dict with the four allowed fields plus email_id. Serialize to JSON.
    # Upload to audit bucket under key f"audit/{email_id}/{entity_type}-{position}.json".
    # IMPORTANT: never include raw_value in the body or the key.
    raise NotImplementedError("Replace with the hash + put_object.")


# Re-run Comprehend so we have offsets to log. Skip the stealth cases for this
# task (they require parsing the Guardrail response shape, which is the next
# task's territory; the audit log here records only the Comprehend pre-stage).
with open(TRACES_PATH) as f:
    rows = list(csv.DictReader(f))

# We need the original text to recover the offsets; pull from the input bucket.
emails_jsonl = s3.get_object(Bucket=INPUT_BUCKET, Key="emails.jsonl")["Body"].read().decode()
emails_by_id = {json.loads(l)["id"]: json.loads(l) for l in emails_jsonl.splitlines() if l.strip()}

entry_count = 0
for r in rows:
    if int(r["comprehend_detected"]) == 0:
        continue
    e = emails_by_id[r["email_id"]]
    ents = comprehend_detect(e["text"])
    for ent in ents:
        raw_value = e["text"][ent["BeginOffset"]:ent["EndOffset"]]
        write_audit_entry(e["id"], ent["Type"], ent["BeginOffset"], raw_value)
        entry_count += 1

print(f"Wrote {entry_count} audit-log objects to {AUDIT_BUCKET}/audit/")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: audit log has the expected schema; no body contains PII patterns.


def checkpoint_3(session):
    # List all audit objects.
    try:
        paginator = s3.get_paginator("list_objects_v2")
        objs = []
        for page in paginator.paginate(Bucket=AUDIT_BUCKET, Prefix="audit/"):
            objs.extend(page.get("Contents", []))
    except ClientError as exc:
        return CheckpointResult(status="fail", error_reason=f"List audit/ failed: {exc!r}")
    if len(objs) < 30:
        return CheckpointResult(
            status="fail",
            error_reason=f"Found {len(objs)} audit objects; expected >= 30 (one per detected entity).",
        )
    for o in objs[:30]:
        body = s3.get_object(Bucket=AUDIT_BUCKET, Key=o["Key"])["Body"].read().decode()
        try:
            entry = json.loads(body)
        except json.JSONDecodeError:
            return CheckpointResult(status="fail", error_reason=f"{o['Key']} body is not JSON.")
        # Schema check.
        if "sha256_hash" not in entry or "entity_type" not in entry or "position" not in entry:
            return CheckpointResult(
                status="fail",
                error_reason=f"{o['Key']} missing required keys; got {list(entry.keys())}.",
            )
        # PII-pattern check on the body.
        for pat in PII_PATTERNS:
            if pat.search(body):
                return CheckpointResult(
                    status="fail",
                    error_reason=f"{o['Key']} body matches a PII regex pattern; the audit log must store hashes, not values.",
                )
    return CheckpointResult(status="pass")


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

The audit log is a chain of evidence: it must prove what was redacted without leaking what was redacted. Hash the value; store the hash. Position + entity type are not sensitive on their own.

</details>

<details><summary>Hint 2 (direction)</summary>

```
def write_audit_entry(email_id, entity_type, position, raw_value):
    h = hashlib.sha256(raw_value.encode()).hexdigest()
    entry = {
        "email_id": email_id,
        "entity_type": entity_type,
        "position": position,
        "sha256_hash": h,
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }
    key = f"audit/{email_id}/{entity_type}-{position}.json"
    s3.put_object(Bucket=AUDIT_BUCKET, Key=key, Body=json.dumps(entry).encode())
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Same as Hint 2. The only subtle gotcha: do not include `raw_value` in the entry dict by accident (e.g., for "easier debugging"); the validator regex-scans the body and will flag any pattern hit.

</details>

**Wallet check.** Audit log writes: a few dozen tiny S3 puts, fractions of a penny. Object lock storage is free (only the underlying S3 storage is billed). Cumulative spend: still under 30 cents.

## Task 4: Confirm tamper-evidence

Object lock in GOVERNANCE mode prevents `DeleteObject` unless the caller has `s3:BypassGovernanceRetention` AND passes `BypassGovernanceRetention=True` on the delete call. The validator:

1. picks one audit object, tries to delete it without the bypass, asserts the operation fails (`AccessDenied`)
2. retries with the bypass, asserts success

You implement both halves below.

In [ ]:
# Task 4: tamper test.

paginator = s3.get_paginator("list_objects_v2")
victim_key = None
victim_version = None
for page in paginator.paginate(Bucket=AUDIT_BUCKET, Prefix="audit/"):
    if page.get("Contents"):
        # We need the version id for object-locked delete.
        v_resp = s3.list_object_versions(Bucket=AUDIT_BUCKET, Prefix=page["Contents"][0]["Key"])
        for v in v_resp.get("Versions", []):
            if v["IsLatest"]:
                victim_key = v["Key"]
                victim_version = v["VersionId"]
                break
        break

print(f"Victim object for tamper test: {victim_key} (version {victim_version})")


def attempt_delete_no_bypass():
    # YOUR CODE: call s3.delete_object(Bucket=AUDIT_BUCKET, Key=victim_key,
    # VersionId=victim_version). The call should raise ClientError with
    # AccessDenied because of governance retention; catch it and return the
    # error code string. If it succeeds (the bucket lock was not enabled), the
    # function should return the string "DELETED" so the validator catches it.
    raise NotImplementedError("Replace with the un-bypassed delete attempt.")


def attempt_delete_with_bypass():
    # YOUR CODE: call s3.delete_object(Bucket=AUDIT_BUCKET, Key=victim_key,
    # VersionId=victim_version, BypassGovernanceRetention=True). Return "DELETED"
    # on success.
    raise NotImplementedError("Replace with the bypassed delete.")


code_no_bypass = attempt_delete_no_bypass()
print(f"Delete without bypass returned: {code_no_bypass}")

code_with_bypass = attempt_delete_with_bypass()
print(f"Delete with bypass returned: {code_with_bypass}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: un-bypassed delete is denied; bypassed delete succeeds.


def checkpoint_4(session):
    if code_no_bypass == "DELETED":
        return CheckpointResult(
            status="fail",
            error_reason=(
                "Un-bypassed delete succeeded. Object lock was not configured correctly at "
                "bucket creation. Lock cannot be enabled retroactively; recreate the audit bucket "
                "with ObjectLockEnabledForBucket=True."
            ),
        )
    if "AccessDenied" not in str(code_no_bypass):
        return CheckpointResult(
            status="fail",
            error_reason=f"Expected AccessDenied on un-bypassed delete, got {code_no_bypass!r}.",
        )
    if code_with_bypass != "DELETED":
        return CheckpointResult(
            status="fail",
            error_reason=f"Bypassed delete should have succeeded, got {code_with_bypass!r}.",
        )
    return CheckpointResult(status="pass")


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

Two `delete_object` calls. Same key, same VersionId. The only difference is the `BypassGovernanceRetention` parameter. Wrap the first call in `try/except ClientError` to capture the AccessDenied response code.

</details>

<details><summary>Hint 2 (direction)</summary>

```
def attempt_delete_no_bypass():
    try:
        s3.delete_object(Bucket=AUDIT_BUCKET, Key=victim_key, VersionId=victim_version)
        return "DELETED"
    except ClientError as exc:
        return exc.response["Error"]["Code"]


def attempt_delete_with_bypass():
    s3.delete_object(
        Bucket=AUDIT_BUCKET,
        Key=victim_key,
        VersionId=victim_version,
        BypassGovernanceRetention=True,
    )
    return "DELETED"
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Same as Hint 2. Verify that the IAM policy attached to your access key includes `s3:BypassGovernanceRetention`; without it the bypass call also denies and the validator reports a permission gap rather than a config gap.

</details>

**Wallet check.** Two `delete_object` calls. Effectively free. Cleanup is next.

## Cleanup

Drop the Bedrock Guardrail, empty + delete both buckets (audit bucket needs the governance bypass for every remaining object version), remove the local CSV. Re-running is safe.

In [ ]:
# NBVAL_SKIP
# Cleanup.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account still holds the audit bucket or the guardrail. Resolve before closing.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: about $0.55.** Comprehend was the biggest line at ~$0.20; Guardrails about $0.15; Haiku ping plus the audit-log puts another ~$0.20. Audit bucket and input bucket dropped. Guardrail config gone. Local traces removed.

## Reflection

These are not graded. They are for you.

1. Your pipeline catches >= 95% of PII. The compliance team asks for 100%. What is one architectural change that gets closer to 100% without rewriting both stages from scratch? Acknowledge that 100% recall on unstructured text is impossible; aim for "asymptotically close at acceptable cost".

2. Audit entries store SHA-256 of the redacted value. An attacker with a known plaintext can verify whether their guess matches a hash (a rainbow-table style attack). What is one mitigation that does not break the replayability requirement that auditors recompute the hash from a fresh sample of the input?

## Exam-Style Practice

**Question 1 (defense in depth):**

A PII pipeline catches 95% of PII at the input stage. What is the right next step for the remaining 5%?

A. Tighten the regex in the input stage.
B. Add an output-stage redaction pass with a different detector.
C. Block all model outputs that look like PII.
D. Manual review of every output.

<details><summary>Show answer</summary>

**Correct: B.**

- A is bounded: regex tightening catches some stealth cases but never all (homoglyph attacks, semantic obfuscation).
- B is correct: defense in depth is the production pattern. The output stage catches what the input stage missed because the model itself may surface PII the input never contained (hallucinated names, synthesized addresses).
- C has prohibitive false-positive cost: any legitimate name or address would be blocked.
- D does not scale; it also creates a new compliance surface (the reviewers see the PII).

</details>

**Question 2 (object lock mode):**

A team enables object lock on the audit bucket in COMPLIANCE mode with a 7-year retention period. Six months later they realize the audit log entries contain a software bug that put PII in the body. What is the most accurate description of their options?

A. Override the retention with the root account and delete the bad entries.
B. The objects cannot be deleted by any user under any conditions until the retention expires.
C. The objects can be deleted by the bucket owner but not by IAM users.
D. The objects can be edited in place to remove the PII.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: COMPLIANCE mode is specifically designed so that NO principal (including root) can shorten the retention period or delete the object. That is the whole point.
- B is correct: COMPLIANCE mode is irreversible until expiry. The lab uses GOVERNANCE mode for exactly this reason: a labs platform cannot leave permanent objects in student accounts.
- C is wrong: object lock is not about IAM identity; it is about the lock itself.
- D is wrong: S3 objects are immutable; "editing in place" means writing a new version, but the old version is still locked.

</details>

**Question 3 (audit log replayability):**

The audit log stores SHA-256 of the redacted value. The compliance auditor asks: "Prove to me that the value at email_017 offset 42 is what your system claims." What is the right protocol?

A. Recompute SHA-256 of the original value (read from the input bucket) and compare to the audit log hash.
B. Decrypt the hash to recover the original value.
C. Look up the value in a separate secrets vault.
D. Ask the user who sent the email to confirm.

<details><summary>Show answer</summary>

**Correct: A.**

- A is correct: SHA-256 is one-way. The auditor independently computes the hash from the source-of-truth input record and confirms it matches what the audit log stored.
- B is wrong: hashes are not encrypted; there is nothing to decrypt.
- C is wrong: a separate value store defeats the purpose of redaction.
- D is wrong: depends on a user response and is not a deterministic audit step.

</details>